# IAN 601 — Module 14: Machine Learning & External Data

This notebook is a bridge into your next machine learning course.

We begin with a formula you already know, then move toward models that learn relationships from data.

**Path:** algebra → data → features → train/test split → prediction → evaluation → pipeline

## 1. Start with algebra: y = 2x + 1

When the rule is known, prediction is straightforward.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

x = np.arange(0, 11)
y = 2 * x + 1

pd.DataFrame({"x": x, "y": y})

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x, y, marker="o")
plt.title("Known Relationship: y = 2x + 1")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 2. Reverse engineer the formula from data

A linear model estimates the slope and intercept from examples.

In [ ]:
from sklearn.linear_model import LinearRegression

X = x.reshape(-1, 1)

linear_model = LinearRegression()
linear_model.fit(X, y)

print("Estimated slope:", linear_model.coef_[0])
print("Estimated intercept:", linear_model.intercept_)

## 3. Real data is noisy

Real observations rarely fall perfectly on one line.

In [ ]:
rng = np.random.default_rng(601)

x_noisy = np.arange(0, 30)
y_noisy = 2 * x_noisy + 1 + rng.normal(0, 5, size=len(x_noisy))

plt.figure(figsize=(7, 4))
plt.scatter(x_noisy, y_noisy)
plt.title("Noisy Data")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 4. Best-fit line

In [ ]:
X_noisy = x_noisy.reshape(-1, 1)

fit_model = LinearRegression()
fit_model.fit(X_noisy, y_noisy)

predicted = fit_model.predict(X_noisy)

plt.figure(figsize=(7, 4))
plt.scatter(x_noisy, y_noisy, label="Observed")
plt.plot(x_noisy, predicted, label="Best-fit line")
plt.title("Observed Data and Best-Fit Line")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

print("Slope:", round(fit_model.coef_[0], 3))
print("Intercept:", round(fit_model.intercept_, 3))

## 5. Load external CSV data

The Module 14 dataset is stored in GitHub and can be read directly in Colab.

In [ ]:
data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module14_ML_Bridge_500.csv"

data = pd.read_csv(data_url)
data.head()

## 6. JSON and API-style data

Many APIs return JSON. Pandas can turn a list of JSON-like records into a DataFrame.

In [ ]:
api_records = [
    {"customer_id": "A1", "visits": 4, "channel": "Web"},
    {"customer_id": "A2", "visits": 7, "channel": "Email"},
    {"customer_id": "A3", "visits": 2, "channel": "Store"}
]

api_df = pd.DataFrame(api_records)
api_df

## 7. Inspect modeling data before fitting

In [ ]:
print("Shape:", data.shape)
print()
print(data.dtypes)
print()
print("Missing values:")
print(data.isna().sum())

In [ ]:
data.describe(include="all").T

## 8. Features become numeric vectors

A model works with feature values. Start with a few numeric columns.

In [ ]:
numeric_features = [
    "visits",
    "avg_session_min",
    "prior_purchases",
    "discount_pct"
]

data[numeric_features].head()

## 9. What if a feature is text?

One-hot encoding creates indicator columns for categories.

In [ ]:
encoded = pd.get_dummies(
    data[["channel", "region"]],
    dtype=int
)

encoded.head()

## 10. Correlation matrix: inspect numeric relationships

In [ ]:
numeric_for_corr = data[
    [
        "visits",
        "avg_session_min",
        "prior_purchases",
        "discount_pct",
        "order_value",
        "converted"
    ]
]

corr = numeric_for_corr.corr()

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(image)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 11. Regression versus classification

- **Regression** predicts a numeric value, such as `order_value`.
- **Classification** predicts a class, such as whether `converted` is 0 or 1.

## 12. Simple regression with several features

In [ ]:
reg_features = data[
    ["visits", "avg_session_min", "prior_purchases", "discount_pct"]
]
reg_target = data["order_value"]

reg_model = LinearRegression()
reg_model.fit(reg_features, reg_target)

pd.Series(
    reg_model.coef_,
    index=reg_features.columns,
    name="coefficient"
)

## 13. Separate features X and target y

For classification, the target is `converted`.

In [ ]:
X = data[
    [
        "visits",
        "avg_session_min",
        "prior_purchases",
        "discount_pct",
        "channel",
        "region"
    ]
]

y = data["converted"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 14. Train/test split

Training data teaches the model. Test data checks how it performs on rows it did not train on.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=601,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 15. Start with a baseline

A baseline gives you something simple to beat.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(np.zeros((len(y_train), 1)), y_train)

baseline_accuracy = baseline.score(
    np.zeros((len(y_test), 1)),
    y_test
)

print("Baseline accuracy:", round(baseline_accuracy, 3))

## 16. Logistic regression: from score to probability

The sigmoid function converts a linear score into a value between 0 and 1.

In [ ]:
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(7, 4))
plt.plot(z, sigmoid)
plt.axhline(0.5, linestyle="--")
plt.title("Sigmoid Curve")
plt.xlabel("Linear score")
plt.ylabel("Probability")
plt.show()

## 17. Preprocess numeric and categorical features

A pipeline keeps preprocessing and modeling steps together.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_cols = [
    "visits",
    "avg_session_min",
    "prior_purchases",
    "discount_pct"
]

categorical_cols = ["channel", "region"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

classifier = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

classifier

## 18. Train the classifier

In [ ]:
classifier.fit(X_train, y_train)

test_accuracy = classifier.score(X_test, y_test)

print("Test accuracy:", round(test_accuracy, 3))

## 19. Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    classifier,
    X_test,
    y_test
)

plt.title("Confusion Matrix")
plt.show()

## 20. Accuracy, precision, recall, and F1

In [ ]:
from sklearn.metrics import classification_report

pred = classifier.predict(X_test)

print(
    classification_report(
        y_test,
        pred,
        digits=3
    )
)

## 21. Probability threshold changes the decision

The default classification threshold is usually 0.5.

In [ ]:
probabilities = classifier.predict_proba(X_test)[:, 1]

threshold = 0.40
pred_40 = (probabilities >= threshold).astype(int)

print("First 10 probabilities:")
print(np.round(probabilities[:10], 3))

print("\nFirst 10 predictions at threshold 0.40:")
print(pred_40[:10])

## 22. Predict a new case

In [ ]:
new_customer = pd.DataFrame({
    "visits": [8],
    "avg_session_min": [7.5],
    "prior_purchases": [2],
    "discount_pct": [10],
    "channel": ["Web"],
    "region": ["West"]
})

prediction = classifier.predict(new_customer)[0]
probability = classifier.predict_proba(new_customer)[0, 1]

print("Predicted class:", prediction)
print("Conversion probability:", round(probability, 3))

## 23. Avoid data leakage

Do not include information that would only be known after the outcome.

For example, if `order_value` is only known after conversion, using it to predict `converted` would leak future information into the model.

## 24. End-to-end machine learning workflow

1. Define the question.
2. Inspect the data.
3. Separate features and target.
4. Split train and test data.
5. Fit preprocessing only through the training workflow.
6. Train a baseline.
7. Train the model.
8. Evaluate on test data.
9. Interpret errors and limitations.
10. Predict new cases carefully.

## Practice challenge

1. Train a regression model to predict `order_value`.
2. Calculate mean absolute error on a regression test split.
3. Change the classification threshold to 0.60.
4. Compare precision and recall before and after the threshold change.
5. Remove one feature and see whether accuracy changes.
6. Explain one possible source of data leakage in your own words.

In [ ]:
# Practice area




## IAN 601 complete

You now have the foundation for the next machine learning course: data preparation, numerical representation, train/test thinking, baseline comparison, evaluation, and reusable pipelines.